# Sıfırdan Dil Modeli · Bölüm 3
## Kayan pencere: girdi ve hedef

Önceki bölümlerde öykümüzü token numaralarına çevirdik. Bir dil modeli bu sayılardan nasıl öğrenir? Tek bir oyun oynayarak: önündeki parçalara bakıp **sıradaki parçayı** tahmin eder. Bu defterde öyküyü, modelin çalışacağı soru-cevap çiftlerine çeviriyoruz: metnin üstünde bir pencere kaydırıyoruz; penceredeki tokenlar **girdi** (`x`), bir sağa kaydırılmışları **hedef** (`y`).

Adımlar Sebastian Raschka'nın *Build a Large Language Model (From Scratch)* kitabının 2.6. bölümünü izler. Kod, kitabın açık kaynak deposundaki kodun ([rasbt/LLMs-from-scratch](https://github.com/rasbt/LLMs-from-scratch), Apache-2.0) Türkçe metin için uyarlamasıdır.

Hücreleri sırayla çalıştırın: hücreye tıklayın, **Shift + Enter**'a basın.

## 1. Önceki bölümlerin kodu ve öykü
Önceki bölümlerde yazdığımız kodu `onceki_bolumler.py` dosyasında topladık (kitaptaki `previous_chapters.py`'nin karşılığı). Onu, öyküyü ve Çırak'ın tokenizer dosyasını depodan indiriyoruz:

In [ ]:
import os
import urllib.request

repo = ("https://raw.githubusercontent.com/"
        "kocahmet1/sifirdan-dil-modeli/main/")
for path in ["onceki_bolumler.py", "veri/kasagi.txt",
             "veri/cirak_tokenizer.json"]:
    file_name = os.path.basename(path)
    if not os.path.exists(file_name):
        urllib.request.urlretrieve(repo + path, file_name)
    print(file_name, os.path.getsize(file_name), "bayt")

`cirak_tokenizer()` BPE bölümünün (Bölüm 2) son hücrelerinde yaptığımızı yapar: Çırak'ın birleştirmelerini okuyup kendi `BPETokenizer` sınıfımıza yükler. Bundan sonra seri boyunca bu tokenizer'ı kullanacağız.

In [ ]:
from onceki_bolumler import cirak_tokenizer

tokenizer = cirak_tokenizer()
print(len(tokenizer.vocab))

Öyküyü okuyup bütünüyle token numaralarına çeviriyoruz:

In [ ]:
with open("kasagi.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

enc_text = tokenizer.encode(raw_text)
print(len(enc_text))

## 2. Girdi ve hedef
Öykünün içinden bir yer seçip 4 tokena bakalım. `context_size` modelin bir seferde gördüğü token sayısı. `x` modelin göreceği **girdi**; `y` tahmin etmesi gereken **hedef**: `x`'in bir token sağa kaydırılmışı.

In [ ]:
enc_sample = enc_text[74:]

context_size = 4
x = enc_sample[:context_size]
y = enc_sample[1:context_size + 1]

print(f"x: {x}")
print(f"y:      {y}")

Tek bir pencerede aslında dört tahmin görevi var: ilk tokendan ikinciyi, ilk ikisinden üçüncüyü, … tahmin etmek. Ok işaretinin solu modelin gördüğü (`context`), sağı tahmin etmesi gereken (`desired`):

In [ ]:
for i in range(1, context_size + 1):
    context = enc_sample[:i]
    desired = enc_sample[i]
    print(context, "---->", desired)

In [ ]:
for i in range(1, context_size + 1):
    context = enc_sample[:i]
    desired = enc_sample[i]
    print(tokenizer.decode(context), "---->", tokenizer.decode([desired]))

## 3. PyTorch: Dataset ve DataLoader
Ara bölümde tanıştığımız PyTorch Colab'da kurulu gelir:

In [ ]:
import torch
print("PyTorch sürümü:", torch.__version__)

Kayan pencereyi bir `Dataset` sınıfına koyuyoruz. `Dataset` PyTorch'un veri kalıbı: kaç örnek olduğunu (`__len__`) ve i'nci örneği (`__getitem__`) söyler. `max_length` pencerenin boyu, `stride` pencerenin her adımda kaç token kayacağı. `torch.tensor` bir sayı listesini tensöre çevirir.

In [ ]:
from torch.utils.data import Dataset, DataLoader


class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        # bütün metni token numaralarına çevir
        token_ids = tokenizer.encode(txt)

        # kayan pencere: max_length boyunda, stride adımla
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]

Veri setini deneyelim: 4 tokenlık pencereler, her adımda 1 token kayarak. `len` kaç pencere olduğunu, `dataset[0]` ilk pencereyi (girdi ve hedef) verir:

In [ ]:
dataset = GPTDatasetV1(raw_text, tokenizer, max_length=4, stride=1)
print(len(dataset))
print(dataset[0])

`DataLoader` veri setinden `batch_size` kadar pencereyi bir araya getirip tek bir tensör yapar: modelin bir seferde işleyeceği paket (*batch*). `shuffle` pencerelerin sırasını karıştırır; `drop_last` son paket eksik kalırsa onu atar.

In [ ]:
def create_dataloader_v1(txt, batch_size=4, max_length=256,
                         stride=128, shuffle=True, drop_last=True,
                         num_workers=0):
    tokenizer = cirak_tokenizer()
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )
    return dataloader

Deneyelim: 4 tokenlık pencereler, her seferinde tek pencere (`batch_size=1`), pencere her adımda 1 token kaysın (`stride=1`). `shuffle=False`: sırayı karıştırma. `iter` ve `next` paketleri tek tek almamızı sağlar.

In [ ]:
dataloader = create_dataloader_v1(
    raw_text, batch_size=1, max_length=4, stride=1, shuffle=False
)

data_iter = iter(dataloader)
first_batch = next(data_iter)
print(first_batch)

In [ ]:
second_batch = next(data_iter)
print(second_batch)

Şimdi pencereler örtüşmesin (`stride=4`) ve bir seferde 8 pencere alalım (`batch_size=8`):

In [ ]:
dataloader = create_dataloader_v1(
    raw_text, batch_size=8, max_length=4, stride=4, shuffle=False
)

data_iter = iter(dataloader)
inputs, targets = next(data_iter)
print("Girdiler:\n", inputs)
print("\nHedefler:\n", targets)

## 4. Gerçek boy
Çırak'ın bağlam uzunluğu 256 token. Aynı öyküden 256'lık pencereler kesersek, `stride`'a göre kaç pencere çıkar?

In [ ]:
for stride in [1, 128, 256]:
    dataset = GPTDatasetV1(raw_text, tokenizer, max_length=256, stride=stride)
    print(f"stride={stride:3}: {len(dataset)} pencere")

In [ ]:
dataloader = create_dataloader_v1(
    raw_text, batch_size=8, max_length=256, stride=256, shuffle=False
)
inputs, targets = next(iter(dataloader))
print(inputs.shape, targets.shape)
print(len(dataloader), "batch")

## Özet
Bir dil modeli sıradaki parçayı tahmin ederek öğrenir; soruyu da cevabı da metnin kendisi verir. Kayan pencere öyküyü girdi (`x`) ve hedef (`y`) çiftlerine böler: `y`, `x`'in bir token kaydırılmışı. `GPTDatasetV1` pencereleri keser, `DataLoader` onları modelin işleyeceği paketlere (*batch*) dizer.

**Kendin dene**
- 4 tokenlık pencerelerle `stride=2` yapın: kaç pencere çıkıyor? (`len(dataset)`)
- `shuffle=True` ile 8'li paketi iki kez alın: ilk paket neden değişiyor?

**Sıradaki bölüm:** gömme (*embedding*): token numaralarından vektörlere.

---
Kaynak: Sebastian Raschka, *Build a Large Language Model (From Scratch)*, Manning 2024, bölüm 2.6. Kod: [rasbt/LLMs-from-scratch](https://github.com/rasbt/LLMs-from-scratch) (Apache-2.0), Türkçe uyarlama: Sıfırdan Dil Modeli. Metin: Ömer Seyfettin, "Kaşağı" (kamu malı). Çırak'ın tokenizer'ı Türkçe Vikipedi metniyle (CC BY-SA 4.0) eğitildi.